# S21 – Concentrations verification

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from S21_comments import Audit, apply_comments   # the code of the comments of the step, apart from the data handling

INPUT = "../../S20_Assay_unification_and_verification/S20.xlsx"
OUTPUT = "../S21.xlsx"
COMMENTS = "../S21_comments.xlsx"
AUDIT = "../S21_audit.xlsx"
STEP = "S21"

df = pd.read_excel(INPUT, keep_default_na=False, na_values=[""]).astype(object)
print(len(df), "records before the step")
audit = Audit(STEP, df, before=INPUT)
put, drop = audit.put, audit.drop                  # a rule sets a value / removes records through the audit, which records the change
changes, removed = audit.changes, audit.removed    # the ledger of the step

3001 records before the step


### 1. One concentration in both units

In [2]:
for i in df.index:
    ug, gl = df.at[i, "Concentration (ug/ml)"], df.at[i, "Concentration (g/l)2"]
    if pd.isna(ug) and not pd.isna(gl):
        put(i, "Concentration (ug/ml)", float(gl) * 1000, f"{STEP}_gl_to_ugml", "unit conversion g/L -> ug/mL (x1000)", outside=False)
    elif not pd.isna(ug) and pd.isna(gl):
        put(i, "Concentration (g/l)2", float(ug) / 1000, f"{STEP}_ugml_to_gl", "unit conversion ug/mL -> g/L (/1000)", outside=False)

### 2. Values from the publication review: the comments of this step

In [3]:
# every value from the publication review: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S21_curated_L1_doi', 'S21_curated_L4_diameter', 'S21_curated_L5_cell', 'S21_curated_L7_row'])

1548 comments applied (S21_curated_L1_doi, S21_curated_L4_diameter, S21_curated_L5_cell, S21_curated_L7_row): 1548 values set, 0 records removed


### 3. Corrections from the check against the publications: the comments of this step

In [4]:
# every correction from the check of this field against the publications: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S21_publication_check_01', 'S21_publication_check_02', 'S21_publication_check_03', 'S21_publication_check_04', 'S21_publication_check_05'])

51 comments applied (S21_publication_check_01, S21_publication_check_02, S21_publication_check_03, S21_publication_check_04, S21_publication_check_05): 51 values set, 0 records removed


### The table after the step and its audit

In [5]:
# the table after the step; the audit module compares it with the table before the step and writes the audit workbook
df.to_excel(OUTPUT, index=False)
audit.write(AUDIT)

0 records removed and 1944 cells changed compared with the table before the step


3001 records after the step | 0 records removed | 2242 cells changed | ../S21_audit.xlsx
